# Imports

In [1]:
%load_ext autoreload
%autoreload 2

# System functionality
import os
import glob 

import config

# Data Analysis
import xarray as xr
import xesmf as xe
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib import colors as mcolors
from matplotlib import ticker as mticker

# Cartopy
from cartopy import crs as ccrs
from cartopy import feature as cfeature
from cartopy import util as cutil
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LongitudeLocator, LatitudeLocator

# Auxiliary Functions
from auxiliary_functions.time_utils import datetime64_to_yyyymmdd, string_to_yyyymm, convert_time_to_ns, convert_ns_to_datetime, extract_years_months
from auxiliary_functions import xarray_utils
from auxiliary_functions.plotting_utils import tick_labeller, set_plot_mode, get_figsize

# Logging
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

In [2]:
# start_date = "2020-12-29T00:00:00.000000000"
# end_date = "2021-02-27T00:00:00.000000000"
# logger.info(f"Start date: {start_date}")
# logger.info(f"End date: {end_date}")

# logger.info("Starting data download script")
# def surface_level_preprocess(ds: xr.Dataset) -> xr.Dataset:
#     # Subset to a specific region and keep only one variable
#     subset_data = ds.sel(
#         # level=pressure_levels,
#         time=ds.time.where(ds['time'].dt.hour.isin([0, 6, 12, 18]), drop=True)
#     )

#     target_grid = xr.Dataset(
#         {
#             "lat": (["lat"], np.arange(-90, 91, 1.0)),
#             "lon": (["lon"], np.arange(0, 360, 1.0)),
#         }
#     )
#     regridder = xe.Regridder(subset_data, target_grid, "bilinear", reuse_weights=False) 

#     return regridder(subset_data) # type: ignore

# def pressure_level_preprocess(ds: xr.Dataset) -> xr.Dataset:
#     # Subset to a specific region and keep only one variable
#     subset_data = ds.sel(
#         level=pressure_levels,
#         time=ds.time.where(ds['time'].dt.hour.isin([0, 6, 12, 18]), drop=True)
#     ).assign_coords({'level': pressure_levels.astype(np.int32)})

#     target_grid = xr.Dataset(
#         {
#             "lat": (["lat"], np.arange(-90, 91, 1.0)),
#             "lon": (["lon"], np.arange(0, 360, 1.0)),
#         }
#     )
#     regridder = xe.Regridder(subset_data, target_grid, "bilinear", reuse_weights=False) 

#     return regridder(subset_data) # type: ignore

# yyyymm_strings = pd.date_range(
#     pd.to_datetime(start_date).to_period("M").to_timestamp(),
#     pd.to_datetime(end_date).to_period("M").to_timestamp(),
#     freq="MS"
# ).strftime("%Y%m")

# logger.info(f"Dates: {np.datetime64(start_date).astype('datetime64[h]')} : {np.datetime64(end_date).astype('datetime64[h]')}")

# graphcast_data_directory = f"/glade/u/home/sressel/spencer-scratch/graphcast_input_data/{string_to_yyyymm(start_date)}_{string_to_yyyymm(end_date)}"
# if not os.path.exists(graphcast_data_directory):
#     logger.info("Creating output directory...")
#     os.makedirs(graphcast_data_directory, exist_ok=True)
# logger.info(f"Graphcast data directory: {graphcast_data_directory}")

# logger.info("Load all data into a single dataset")
# all_data = xr.open_mfdataset(f"{graphcast_data_directory}/*.nc")
# logger.info(f"    Saving data...")
# all_data.to_netcdf(f"{graphcast_data_directory}/era5_data.nc")
# logger.info("Finished")

In [3]:
years, months = extract_years_months(start_date, end_date)

# Surface level variables
logger.info("Surface level variables")
surface_base = "/gdex/data/d633000/e5.oper.an.sfc"

surface_variables = {
    "2m_temperature": "2t",
    "mean_sea_level_pressure": "msl",
    "10m_u_component_of_wind": "10u",
    "10m_v_component_of_wind": "10v"
}

surface_variables_old_names = {
    "2m_temperature": "VAR_2T",
    "mean_sea_level_pressure": "MSL",
    "10m_u_component_of_wind": "VAR_10U",
    "10m_v_component_of_wind": "VAR_10V"
}

for variable in surface_variables.keys():
    files_list = []

    logger.info(f"  {variable}")
    for ym in yyyymm_strings:
        pattern = f"{surface_base}/{ym}/e5.oper.an.sfc.*_{surface_variables[variable]}.*.nc"
        files_list.extend(glob.glob(pattern))


NameError: name 'start_date' is not defined

# Load RMM data

In [ ]:
def load_txt_as_xarray(path):
    # Read the whitespace-delimited file
    df = pd.read_csv(
        path,
        delim_whitespace=True,
        header=None,
        names=["year", "month", "day", "hour", "var1", "var2", "var3"],
        na_values=[-99.0],
    )

    # Build datetime64 index
    df["time"] = pd.to_datetime(df[["year", "month", "day", "hour"]])

    # Set time as index
    df = df.set_index("time")

    # Drop the original date columns
    df = df.drop(columns=["year", "month", "day", "hour"])

    # Convert to xarray Dataset
    ds = xr.Dataset.from_dataframe(df)

    return ds

# Example usage:
RMM_indices = load_txt_as_xarray("RMM.txt")
RMM1 = RMM_indices["var1"]
RMM2 = RMM_indices["var2"]
amplitude = RMM_indices["var3"]


In [ ]:
def convert_time_to_ns(ds):
    # Original datetime64 coordinate
    datetime = ds["time"]

    # Compute nanoseconds since first timestep
    t0 = datetime.values[0]
    time_ns = (datetime.values - t0).astype("timedelta64[ns]").astype("timedelta64[ns]")

    new_datetime = datetime.expand_dims('batch').assign_coords(
        datetime=("time", datetime.values),   # secondary coordinate
        time=("time", time_ns)                # replace primary coordinate
    )

    # Assign new coordinates
    ds = ds.expand_dims('batch').assign_coords(
        datetime=new_datetime,
        time=("time", time_ns)
    )

    return ds

In [ ]:
start_date = '1992-08-14T00:00:00.000000000'
end_date = '1992-11-12T00:00:00.000000000'
yyyymm_strings = pd.date_range(
    pd.to_datetime(start_date).to_period("M").to_timestamp(),
    pd.to_datetime(end_date).to_period("M").to_timestamp(),
    freq="MS"
).strftime("%Y%m")


def pressure_level_preprocess(ds: xr.Dataset) -> xr.Dataset:
    # Subset to a specific region and keep only one variable
    subset_data = ds.sel(
        level=pressure_levels,
        time=ds.time.where(ds['time'].dt.hour.isin([0, 6, 12, 18]), drop=True)
    ).assign_coords({'level': pressure_levels.astype(np.int32)})

    target_grid = xr.Dataset(
        {
            "lat": (["lat"], np.arange(-90, 91, 1.0)),
            "lon": (["lon"], np.arange(0, 360, 1.0)),
        }
    )
    regridder = xe.Regridder(subset_data, target_grid, "bilinear", reuse_weights=False) 

    return regridder(subset_data) # type: ignore

pressure_level_base = "/gdex/data/d633000/e5.oper.an.pl"

pressure_levels = xr.DataArray(
    data = [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000],
    dims=['level'],
    coords={'level': [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]}
)

pressure_level_variables = {
    "geopotential": "z",
    # "temperature": "t",
    # "u_component_of_wind": "u",
    # "v_component_of_wind": "v",
    # "specific_humidity": "q",
    # "vertical_velocity": "w",
}

pressure_level_variables_old_names = {
    "geopotential": "Z",
    "temperature": "T",
    "u_component_of_wind": "U",
    "v_component_of_wind": "V",
    "specific_humidity": "Q",
    "vertical_velocity": "W",
}

for variable in pressure_level_variables.keys():
    files_list = []

    logger.info(f"  {variable}")
    for ym in yyyymm_strings:
        pattern = f"{pressure_level_base}/{ym}/e5.oper.an.pl.*_{pressure_level_variables[variable]}.*.nc"
        files_list.extend(glob.glob(pattern))

    if not files_list:
        logger.info(f"No files found for variable {variable} in month {ym}")
    else:
        logger.info(f"    Loading files...")
        pressure_level_data = xr.open_mfdataset(sorted(files_list)[:3], preprocess=pressure_level_preprocess).load()


In [ ]:
# yyyymm_strings
pressure_level_data.coords['time']
# sorted(files_list)

# xr.open_dataset("/gdex/data/d633000/e5.oper.an.pl/199208/e5.oper.an.pl.128_130_t.ll025sc.1992080100_1992080123.nc")

In [ ]:
target_duration

In [ ]:
# plt.scatter(
#     RMM1.isel(time=slice(3371-100, 3371+100)),
#     RMM2.isel(time=slice(3371-100, 3371+100)),
# )
# plt.xlim(-4, 4)
# plt.ylim(-4, 4)
# plt.gca().set_aspect('equal')

# Configure plot
[fig, ax] = plt.subplots(figsize=(9,9))
plt.rcParams["axes.edgecolor"] = "black"
plt.rcParams["axes.linewidth"] = 3
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
plt.xlabel("RMM1")
plt.ylabel("RMM2")
ax.set_facecolor("white")

# Plot index points
colormap = sns.color_palette("viridis", as_cmap=True)
start_time = '2009-10-01T00:00:00.000000000'
end_time = '2010-04-30T00:00:00.000000000'
start_index = list(amplitude.time.values).index(amplitude.sel(time=start_time).time)
end_index = list(amplitude.time.values).index(amplitude.sel(time=end_time).time)
ax.plot(RMM1[start_index], RMM2[start_index], color="black", marker=".", ls="-", ms=30)
for i in range(start_index + 1, end_index + 1):
    ax.plot(
        RMM1[i],
        RMM2[i],
        color=colormap((i - start_index) / (end_index - start_index)),
        marker="o",
        ms=10,
    )

# Add phase regions overlay
circle1 = plt.Circle((0, 0), 1.0, color="k", fill=False, lw=3, zorder=10)
ax.hlines(y=0, xmin=-4, xmax=-1, color="k", lw=3, ls="-")
ax.hlines(y=0, xmin=1, xmax=4, color="k", lw=3, ls="-")
ax.vlines(x=0, ymin=-4, ymax=-1, color="k", lw=3, ls="-")
ax.vlines(x=0, ymin=1, ymax=4, color="k", lw=3, ls="-")
ax.plot([np.sqrt(2) / 2, 4], [np.sqrt(2) / 2, 4], color="k", lw=3, ls="-")


x_vals = {
    1:-1,
    2:-1,
    3:1,
    4:1,
    5:1,
    6:1,
    7:-1,
    8:-1
}

y_val1 = {
    1:0,
    2:-4,
    3:-4,
    4:0,
    5:0,
    6:np.linspace(0,4,len(RMM1)),
    7:np.linspace(0,4,len(RMM1)),
    8:0
}

y_val2 = {
    1:-1*np.linspace(0, 4, len(RMM1)),
    2:-1*np.linspace(0, 4, len(RMM1)),
    3:-1*np.linspace(0, 4, len(RMM1)),
    4:-1*np.linspace(0, 4, len(RMM1)),
    5:1*np.linspace(0, 4, len(RMM1)),
    6:4,
    7:4,
    8:1*np.linspace(0, 4, len(RMM1))
}

# Fill one of the phases in with blue
# val=2
# ax.fill_between(
#     x_vals[val]*np.linspace(0, 4, len(RMM1)), 
#     y_val1[val], 
#     y_val2[val]
# )

# x = np.linspace(-1,1,100)
# ax.fill_between(x, -np.sqrt(1-x**2), np.sqrt(1-x**2), color='white')

# Add lines to differentiate the phases
ax.plot([np.sqrt(2) / 2, 4], [-np.sqrt(2) / 2, -4], color="k", lw=3, ls="-")
ax.plot([-4, -np.sqrt(2) / 2], [4, np.sqrt(2) / 2], color="k", lw=3, ls="-")
ax.plot([-4, -np.sqrt(2) / 2], [-4, -np.sqrt(2) / 2], color="k", lw=3, ls="-")
ax.add_patch(circle1)

# Add phase labels
ax.text(-3.5, -0.26, f'Phase 1',  horizontalalignment='center',
     verticalalignment='center')
ax.text(-0.51, -3.75, f'Phase 2', horizontalalignment='center',
     verticalalignment='center')
ax.text(0.5, -3.75, f'Phase 3',   horizontalalignment='center',
     verticalalignment='center')
ax.text(3.5, -0.26, f'Phase 4',   horizontalalignment='center',
     verticalalignment='center')
ax.text(3.5, 0.25, f'Phase 5',    horizontalalignment='center',
     verticalalignment='center')
ax.text(0.5, 3.75, f'Phase 6',    horizontalalignment='center',
     verticalalignment='center')
ax.text(-0.51, 3.75, f'Phase 7',  horizontalalignment='center',
     verticalalignment='center')
ax.text(-3.5, 0.25, f'Phase 8',   horizontalalignment='center',
     verticalalignment='center')

# Add MJO-location labels
# ax.text(0, 3.5, f'Western Pacific',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(-3.3, 0, f'Western Hemisphere \n & Africa',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(3.3, 0, f'Maritime Continent',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(0, -3.5, f'Indian Ocean',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

ax.set_aspect("equal")
plt.tight_layout()

plt.show()

# Process ERA5 data

In [ ]:
# # start_time = "2021-05-01"
# # end_time = "2022-12-31"

# start_time = "1974-00-01"
# end_time = "1979-12-31"

# variable = 'zonal_wind'

# variable_name = {
#     'zonal_wind':'u_component_of_wind',
#     'meridional_wind':'v_component_of_wind'
#     }

# from datetime import datetime
# from dateutil.relativedelta import relativedelta

# def iter_year_months(start, end):
#     """
#     Yield (year, month) pairs for all months between two arbitrary dates.
#     Accepts datetime objects or ISO-format strings.
#     """

#     # Parse strings if needed
#     if isinstance(start, str):
#         start = datetime.fromisoformat(start)
#     if isinstance(end, str):
#         end = datetime.fromisoformat(end)

#     # Normalize to first-of-month
#     cursor = start.replace(day=1)
#     end_month = end.replace(day=1)

#     while cursor <= end_month:
#         yield cursor.year, cursor.month
#         cursor += relativedelta(months=1)

# import cdsapi

# c = cdsapi.Client()

# for year, month in iter_year_months(start_time, end_time):
#     print(year, month)
#     output_file = f"/glade/derecho/scratch/sressel/ECMWF/ERA5/daily_data/daily_25_degree_{variable}_{year}_{month}_raw.nc"
#     c.retrieve(
#         "reanalysis-era5-pressure-levels",
#         {
#             "product_type": "reanalysis",
#             "variable": [
#                 variable_name[variable],
#                 # "v_component_of_wind",
#             ],
#             "pressure_level": [
#                 "100", "125", "150", "175", "200", "225", "250", "300", "350", "400",
#                 "450", "500", "550", "600", "650", "700", "750", "775", "800", "825",
#                 "850", "875", "900", "925", "950", "975", "1000",
#             ],
#             "year": year,
#             "month": month,
#             "day": [
#                 "01","02","03","04","05","06","07","08","09","10",
#                 "11","12","13","14","15","16","17","18","19","20",
#                 "21","22","23","24","25","26","27","28","29","30","31",
#             ],
#             "time": ["00:00"],   # daily at 00 UTC
#             "area": [30, -180, -30, 180],  # North, West, South, East
#             "grid": [2.5, 2.5],  # 2.5° x 2.5°
#             "format": "netcdf",
#         },
#         output_file,
#     )

# # data_processed = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/ECMWF/ERA5/daily_data/daily_25_degree_{variable}_{start_time.year}_{end_time.year}_raw.nc").rename(
# #     {
# #         'longitude': 'lon',
# #         'latitude': 'lat',
# #         'valid_time': 'time',
# #         'pressure_level': 'plev',
# #     }
# # ).transpose("time", "plev", "lat", "lon").isel(plev=slice(None, None, -1)).sortby('plev').sortby('lat').coord_funcs.lon_to_360('lon')
# # data_processed.to_netcdf("/glade/u/home/sressel/spencer-scratch/ECMWF/ERA5/daily_data/daily_25_degree_{variable}_{start_time.year}_{end_time.year}.nc")


# Load Graphcast sample data

In [ ]:
gc_data = xr.open_dataset("/glade/u/home/sressel/thesis-work/python/graphcast/gc-initial-condition-optimization/data/era5_sample.nc")
gc_data.coords['datetime']

# Analyze optimized data

In [8]:
import xarray as xr
init_date = "2021-01-01T00:00:00"
input_data = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/202012_202102/era5_data.nc").drop_vars(['number', 'expver'])

input_data = input_data.assign_coords({'datetime':input_data.datetime.sel(batch=0, drop=True)}).swap_dims({'time':'datetime'}).sel(batch=0, drop=True)

In [16]:


test = xr.zeros_like(xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/202012_202102/era5_data.nc").isel(time=[0,1]))
test

<xarray.Dataset> Size: 44MB
Dimensions:                       (batch: 1, time: 2, lat: 181, lon: 360,
                                   level: 13)
Coordinates:
  * lat                           (lat) float64 1kB -90.0 -89.0 ... 89.0 90.0
  * lon                           (lon) float64 3kB 0.0 1.0 2.0 ... 358.0 359.0
    datetime                      (batch, time) datetime64[ns] 16B ...
  * time                          (time) timedelta64[ns] 16B 00:00:00 06:00:00
  * level                         (level) int32 52B 50 100 150 ... 850 925 1000
    number                        int64 8B ...
    expver                        (time) <U4 32B ...
Dimensions without coordinates: batch
Data variables: (12/14)
    10m_u_component_of_wind       (batch, time, lat, lon) float32 521kB 0.0 ....
    10m_v_component_of_wind       (batch, time, lat, lon) float32 521kB 0.0 ....
    2m_temperature                (batch, time, lat, lon) float32 521kB 0.0 ....
    geopotential                  (batch, time, level, lat, lon) float32 7MB ...
    geopotential_at_surface       (lat, lon) float32 261kB 0.0 0.0 ... 0.0 0.0
    land_sea_mask                 (lat, lon) float32 261kB 0.0 0.0 ... 0.0 0.0
    ...                            ...
    temperature                   (batch, time, level, lat, lon) float32 7MB ...
    toa_incident_solar_radiation  (batch, time, lat, lon) float32 521kB 0.0 ....
    total_precipitation_6hr       (batch, time, lat, lon) float32 521kB 0.0 ....
    u_component_of_wind           (batch, time, level, lat, lon) float32 7MB ...
    v_component_of_wind           (batch, time, level, lat, lon) float32 7MB ...
    vertical_velocity             (batch, time, level, lat, lon) float32 7MB ...
Attributes:
    regrid_method:  bilinear

In [6]:
optimization_initial_date = "2021-01-01T00"
optimization_timesteps = 60

# optimized_data_array = xr.zeros_like(input_data.isel(time=0, drop=True))
optimized_data_file = glob.glob(
    f"{config.GRAPHCAST_DATA_DIRECTORY}/perfect_model_params/{optimization_initial_date}/{optimization_initial_date}_*_{optimization_timesteps}.npz"
)
optimized_data = np.load(optimized_data_file[0])
optimized_data

NpzFile '/glade/u/home/sressel/spencer-scratch/graphcast_output/perfect_model_params/2021-01-01T00/2021-01-01T00_94_60.npz' with keys: 10m_u_component_of_wind, 10m_v_component_of_wind, 2m_temperature, day_progress_cos, day_progress_sin...

In [17]:
optimized_data_array = test
# Assign variables with inferred dims
for key in [
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    '2m_temperature',
    # 'day_progress_cos',
    # 'day_progress_sin',
    'geopotential',
    'geopotential_at_surface',
    'land_sea_mask',
    'mean_sea_level_pressure',
    'specific_humidity',
    'temperature',
    'toa_incident_solar_radiation',
    'total_precipitation_6hr',
    'u_component_of_wind',
    'v_component_of_wind',
    'vertical_velocity'
]:

    arr = optimized_data[key]

    if arr.ndim == 2:
        optimized_data_array[key] = (("lat", "lon"), arr)

    elif arr.ndim == 3:
        optimized_data_array[key] = (("time", "lat", "lon"), arr)

    elif arr.ndim == 4:
        optimized_data_array[key] = (("batch", "time", "lat", "lon"), arr)

    elif arr.ndim == 5:
        optimized_data_array[key] = (("batch", "time", "level", "lat", "lon"), arr)

    else:
        raise ValueError(f"Don't know how to assign dims for {key} with shape {arr.shape}")

In [ ]:
def datetime_to_ns(initial_time, final_time):
    return (final_time.astype("timedelta64[ns]") - initial_time.astype("timedelta64[ns]")).astype("timedelta64[ns]")

In [ ]:
start_date = '2009-12-01T00:00:00.000000000'
end_date = '2010-01-31T00:00:00.000000000'

optimized_day = datetime_to_ns(
    np.datetime64(start_date), np.datetime64('2009-12-26T00:00:00.000000000')
)
level_to_plot = 850
variable_to_plot = 'specific_humidity'

# [fig, ax] = plt.subplots(3, 1, figsize=(12,16))

fig, ax = plt.subplots(nrows=3,ncols=1,
                        subplot_kw={'projection': ccrs.PlateCarree()},
                        figsize=(11,8.5))

i  =0
c_opt = ax[0].contourf(
    optimized_data_array[variable_to_plot].lon,
    optimized_data_array[variable_to_plot].lat,
    optimized_data_array[variable_to_plot].sel(level=level_to_plot).isel(batch=0, time=i),
    levels = 21
)
fig.colorbar(c_opt, ax=ax[0])
ax[0].add_feature(cartopy.feature.COASTLINE, linewidth=0.5)

c_init = ax[1].contourf(
    input_data.lon,
    input_data.lat,
    input_data[variable_to_plot].sel(time=optimized_day, level=level_to_plot).isel(batch=0),
    levels = c_opt.levels
)
fig.colorbar(c_init, ax=ax[1])
ax[1].add_feature(cartopy.feature.COASTLINE, linewidth=0.5)

c_diff = ax[2].contourf(
    input_data.lon,
    input_data.lat,
    (optimized_data_array[variable_to_plot].sel(level=level_to_plot).isel(time=i, batch=0) - input_data[variable_to_plot].sel(time=optimized_day, level=level_to_plot).isel(batch=0)),
    # levels = np.linspace(-0.01, 0.01, 21),
    levels=21,
    cmap = 'coolwarm'
)
fig.colorbar(c_diff, ax=ax[2])
ax[2].add_feature(cartopy.feature.COASTLINE, linewidth=0.5)

# for axis in ax:
#     rect = patches.Rectangle(
#         (70, -10),        # lower-left corner
#         30, 20,   # rectangle size
#         linewidth=1.5,
#         edgecolor='red',
#         facecolor='none',   # or a color like 'lightgray'
#         alpha=0.5
#     )

#     axis.add_patch(rect)

for axis in ax:
    axis.set_xlim(60, 110)
    axis.set_ylim(-15, 15)

plt.show()

# Analyze predicted data

In [14]:
logger.info("Load Graphcast Prediction Data")
prediction_initial_date = "2021-01-01T00"
prediction_timesteps = 120

logger.info(f"Initial Date: {prediction_initial_date}, {prediction_timesteps//config.TIMESTEPS_PER_DAY} day prediction")
prediction_filepath = f"{config.GRAPHCAST_DATA_DIRECTORY}/perfect_model_forecasts/{prediction_initial_date}/{prediction_initial_date}_{prediction_timesteps}.zarr"

init_date = prediction_filepath.split("/")[-1].split("_")[0]

logger.info("Loading predicted data...")
predicted_data = xr.open_zarr(prediction_filepath).sel(batch=0, drop=True).drop_vars(['number', 'expver'])
datetimes = convert_ns_to_datetime(predicted_data.time, init_date)
predicted_data = predicted_data.assign_coords({'datetime':datetimes.datetime}).swap_dims({'time':'datetime'})

logger.info("Loading initial condition data...")
# initial_conditions_filepath = str(predicted_data['initial_conditions_path'].values)
# input_data = xr.open_dataset(initial_conditions_filepath).drop_vars(['number', 'expver'])
input_data = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/202012_202102/era5_data.nc").drop_vars(['number', 'expver'])

input_data = input_data.assign_coords({'datetime':input_data.datetime.sel(batch=0, drop=True)}).swap_dims({'time':'datetime'}).sel(batch=0, drop=True)

logger.info("Finished")

2026-06-12 15:19:58,293 [INFO] Load Graphcast Prediction Data
2026-06-12 15:19:58,295 [INFO] Initial Date: 2021-01-01T00, 30 day prediction
2026-06-12 15:19:58,295 [INFO] Loading predicted data...


2026-06-12 15:19:58,327 [INFO] Loading initial condition data...
/glade/derecho/scratch/sressel/tmp/ipykernel_125578/3013781216.py:18: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  input_data = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/202012_202102/era5_data.nc").drop_vars(['number', 'expver'])
2026-06-12 15:19:58,432 [INFO] Finished


In [ ]:
level_to_plot = 850
variable_to_plot = 'specific_humidity'
unit_multiplier = 1000

fig, ax = plt.subplots(nrows=3,ncols=1,
                        subplot_kw={'projection': ccrs.PlateCarree()},
                        figsize=(11,8.5))

i=predicted_data.datetime[1]
c_input = ax[0].contourf(
    input_data[variable_to_plot].lon,
    input_data[variable_to_plot].lat,
    (unit_multiplier*input_data[variable_to_plot]).sel(level=level_to_plot).sel(datetime=i),
    levels = 21
)
fig.colorbar(c_input, ax=ax[0])
ax[0].add_feature(cfeature.COASTLINE, linewidth=0.5)

c_pred = ax[1].contourf(
    predicted_data.lon,
    predicted_data.lat,
    (unit_multiplier*predicted_data[variable_to_plot]).sel(level=level_to_plot).sel(datetime=i),
    levels = c_input.levels
)
fig.colorbar(c_pred, ax=ax[1])
ax[1].add_feature(cfeature.COASTLINE, linewidth=0.5)

c_diff = ax[2].contourf(
    input_data.lon,
    input_data.lat,
    unit_multiplier*(
        predicted_data[variable_to_plot].sel(datetime=i)
        - input_data[variable_to_plot].sel(datetime=i)
    ).sel(level=level_to_plot),
    # levels = np.linspace(-0.01, 0.01, 21),
    levels=21,
    cmap = 'coolwarm'
)
fig.colorbar(c_diff, ax=ax[2])
ax[2].add_feature(cfeature.COASTLINE, linewidth=0.5)

for axis in ax:
    axis.set_xlim(60, 110)
    axis.set_ylim(-15, 15)

# for axis in ax:
#     rect = patches.Rectangle(
#         (70, -10),        # lower-left corner
#         30, 20,   # rectangle size
#         linewidth=1.5,
#         edgecolor='red',
#         facecolor='none',   # or a color like 'lightgray'
#         alpha=0.5
#     )

#     axis.add_patch(rect)
plt.show()

## Plot Hovmoller

In [ ]:
input_data_subset = input_data.sel(
    datetime=slice(
        predicted_data.datetime[0],
        predicted_data.datetime[-1]
    )
)

variable_to_plot = 'total_precipitation_6hr'
unit_multiplier = (1000/6)

SAVE_FIG = False
PLOT_MODE = "slides"
FIG_LAYOUT = 'full'
plt.style.use('bmh')
set_plot_mode(PLOT_MODE)
plt.rcParams['mathtext.fontset'] = 'dejavusans'
plt.rcParams['font.size'] = '14'
plt.rcParams['axes.labelsize'] = '14'
plt.rcParams['xtick.labelsize'] = '12'
plt.rcParams['ytick.labelsize'] = '12'
plt.rcParams['lines.linewidth'] = '2'

fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))

gs = GridSpec(2, 3, figure=fig, height_ratios=[30,1])
gs.update(top=1, bottom=0, left=0, right=1, wspace=0.1)

axes = [
    fig.add_subplot(gs[0,0]),
    fig.add_subplot(gs[0,1]),
    fig.add_subplot(gs[0,2]),
]
cbar_axes = [
    fig.add_subplot(gs[1, :2]),
    fig.add_subplot(gs[1, -1])
]

im_era5 = axes[0].contourf(
    input_data_subset[variable_to_plot].lon,
    input_data_subset[variable_to_plot].datetime,
    unit_multiplier*(input_data_subset[variable_to_plot].sel(lat=slice(-10,10)).mean(dim='lat')),
    levels=np.arange(0, 2.25, 0.25)
)
im_pred = axes[1].contourf(
    predicted_data.lon,
    predicted_data.datetime,
    unit_multiplier*(predicted_data[variable_to_plot].sel(lat=slice(-10,10)).mean(dim='lat')),
    levels=im_era5.levels
)
fig.colorbar(im_era5, cax=cbar_axes[0], orientation='horizontal')
im_diff = axes[2].contourf(
    input_data_subset[variable_to_plot].lon,
    input_data_subset[variable_to_plot].datetime,
    unit_multiplier*(
        input_data_subset[variable_to_plot].sel(lat=slice(-10,10)).mean(dim='lat')
        - predicted_data[variable_to_plot].sel(lat=slice(-10,10)).mean(dim='lat')
    ),
    cmap='coolwarm',
    levels=np.arange(-2, 2.5, 0.5),
    norm=mcolors.CenteredNorm(vcenter=0)
)
fig.colorbar(im_diff, cax=cbar_axes[1], orientation='horizontal')

for index, ax in enumerate(axes):
    ax.set_yticks(input_data_subset.datetime[::4*7])
    if index != 0:
        ax.set_yticklabels('')
    ax.set_xticks(np.arange(0, 360, 60), labels=tick_labeller(np.arange(0, 360, 60), 'lon'))
    ax.grid(axis='y', alpha=0.5)
plt.show()

## Plot horizontal structures

In [ ]:
variable_to_plot = 'total_precipitation_6hr'
unit_multiplier = (1000/6)
variable_units = r"mm hr$^{-1}$"

# variable_to_plot = 'specific_humidity'
# unit_multiplier = (1000)
# variable_units = r"g kg$^{-1}$"
# plev_to_plot = 850

time_to_plot = np.datetime64(init_date) + np.timedelta64(5, 'D')


SAVE_FIG = False
PLOT_MODE = "slides"
FIG_LAYOUT = 'full'
plt.style.use('bmh')
set_plot_mode(PLOT_MODE)


fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))
gs = GridSpec(3, 2, figure=fig, width_ratios=[30, 1])
gs.update(top=1, bottom=0, left=0, right=1, hspace=0.35, wspace=0.1)

central_longitude = 180
proj = ccrs.PlateCarree(central_longitude=central_longitude)
data_crs = ccrs.PlateCarree()
coastline_width = 1


ax = [
    fig.add_subplot(gs[0,0], projection=proj),
    fig.add_subplot(gs[1,0], projection=proj),
    fig.add_subplot(gs[2,0], projection=proj),
]
cbar_ax = [
    fig.add_subplot(gs[:2,-1]),
    fig.add_subplot(gs[2,-1])
]

cdata_input = xarray_utils.add_cyclic_point(
    input_data[variable_to_plot].coord_funcs.sel(level=850),
    dim='lon'
)

cdata_pred = xarray_utils.add_cyclic_point(
    predicted_data[variable_to_plot].coord_funcs.sel(level=850),
    dim='lon'
)

ax[0].set_title("a) ERA5", loc='left')
im = ax[0].contourf(
    cdata_input.lon,
    cdata_input.lat,
    (unit_multiplier*cdata_input).sel(datetime=time_to_plot),
    levels=np.arange(0, 5.5, 0.5), 
    # levels=np.linspace(-21, 21, 21),
    transform=data_crs,
    cmap='viridis', 
    extend='max'
)

ax[1].set_title("b) Graphcast", loc='left')
ax[1].contourf(
    cdata_pred.lon,
    cdata_pred.lat,
    (unit_multiplier*cdata_pred).sel(datetime=time_to_plot),
    # levels=np.arange(0, 27.5, 2.5), 
    levels=im.levels, 
    transform=data_crs,
    cmap='viridis', 
    extend='max'
)
# Add colorbar
cbar = fig.colorbar(im, cax=cbar_ax[0])
cbar.set_label(f"{variable_units}")

ax[2].set_title("c) ERA5 - Graphcast", loc='left')
im_diff = ax[2].contourf(
    cdata_pred.lon,
    cdata_pred.lat,
    (
        (unit_multiplier*cdata_pred).sel(datetime=time_to_plot)
         - (unit_multiplier*cdata_input).sel(datetime=time_to_plot)
    ),
    levels=np.linspace(-16, 3, 11), 
    transform=data_crs,
    cmap='coolwarm', 
    norm=mcolors.CenteredNorm(vcenter=0),
    extend='both'
)
cbar = fig.colorbar(im_diff, cax=cbar_ax[1])
cbar.set_label(f"{variable_units}")

for axis in ax:
    # axis.set_aspect('equal')
    axis.add_feature(cfeature.COASTLINE, lw=coastline_width)
    minlon = central_longitude - 120
    maxlon = central_longitude
    axis.set_extent([minlon, maxlon, -30, 30], data_crs)
    gl = axis.gridlines(
        crs=data_crs,
        draw_labels=True,
        linewidth=1,
        color="gray",
        alpha=0.75,
        linestyle="-",
        zorder=15
    )
    gl.right_labels = False
    gl.top_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(60,210,15))
    gl.xformatter = LongitudeFormatter()
    gl.ylocator = mticker.FixedLocator(np.arange(-30,45,15))
    gl.yformatter = LatitudeFormatter()

fig.suptitle(
    f"{np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%d-%^b-%Y')}",
    x=ax[0].get_position().x0 + 0.5 * ax[0].get_position().width,
    y=1.05,
    ha="center"
)

output_filename = f"ERA5 vs. Graphcast Horizontal Structures - {np.datetime64(time_to_plot).astype('M8[ms]').astype('O').strftime('%Y-%m-%d-T%H')}.svg"
logger.info(f"Output directory: {config.OUTPUT_DIRECTORY}/")
logger.info(f"Output filename: {output_filename}")
if SAVE_FIG:
    logger.info("Saving...")
    plt.savefig(
        f"{config.OUTPUT_DIRECTORY}/{output_filename}.svg",
        dpi=500,
        bbox_inches="tight",
    )
else:
    logger.info("Not Saving")
logger.info("Finished")

plt.show()

In [ ]:
variable_to_plot = 'total_precipitation_6hr'
plev_to_plot = 850

times_to_plot = np.arange(
    np.datetime64(init_date) + np.timedelta64(1, 'D'),
    np.datetime64(init_date) + np.timedelta64(21, 'D'),
    np.timedelta64(2, 'D')
    )

fig = plt.figure(figsize=(12, len(times_to_plot)))
gs = GridSpec(len(times_to_plot), 2, figure=fig, width_ratios=[30, 1])
gs.update(top=1, bottom=0, left=0, right=0.8, hspace=0.1, wspace=0.05)

proj = ccrs.PlateCarree(central_longitude=180)
data_crs = ccrs.PlateCarree()

ax = [
    fig.add_subplot(gs[i, 0], projection=proj) 
    for i in range(len(times_to_plot))
]

cbar_axis = fig.add_subplot(gs[:, 1])

cf_data = xarray_utils.add_cyclic_point(
    predicted_data[variable_to_plot].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
).stats.standardize(dim=['lat', 'lon'])

quiv_u_data = xarray_utils.add_cyclic_point(
    predicted_data['u_component_of_wind'].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
)

quiv_v_data = xarray_utils.add_cyclic_point(
    predicted_data['v_component_of_wind'].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
)

# ax[0].set_title(f"Event starting {start_time.astype('M8[ms]').astype('O').strftime('%d%^b%Y')}", fontsize=16)
for index, time in enumerate(times_to_plot):
    im = ax[index].contourf(
        cf_data.lon,
        cf_data.lat,
        cf_data.sel(datetime=time),
        # levels=np.arange(-125, 125, 25),
        levels=21,
        transform=data_crs,
        cmap='coolwarm',
        norm=mcolors.CenteredNorm(vcenter=0),
    )
    ax[index].add_feature(cfeature.COASTLINE, lw=1)
    # ax[index].text(
    #     s=f"{(time-primary_event_max_times[event]).values.astype('timedelta64[D]')}", 
    #     x=1-0.015, 
    #     y=0.9,
    #     transform=ax[index].transAxes,
    #     fontsize=12,
    #     verticalalignment='top',
    #     horizontalalignment='right',
    #     bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none')
    #     )

    arrow_spacing = 4
    ax[index].quiver(
        quiv_u_data.lon[::2*arrow_spacing],
        quiv_v_data.lat[::arrow_spacing],
        quiv_u_data.sel(datetime=time)[::arrow_spacing, ::2*arrow_spacing].values,
        quiv_v_data.sel(datetime=time)[::arrow_spacing, ::2*arrow_spacing].values,
        transform=data_crs,
        width=0.002,
        # scale=200
    )

    minlon = central_longitude - 180
    maxlon = central_longitude + 180
    ax[index].set_extent([minlon, maxlon, -30, 30], data_crs)
    gl = ax[index].gridlines(
        crs=data_crs,
        draw_labels=(True if index == len(times_to_plot) - 1 else False),
        linewidth=1,
        color="gray",
        alpha=0.75,
        linestyle="-",
        zorder=15

    )
    gl.right_labels = False
    gl.top_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180,180,60))
    gl.xformatter = LongitudeFormatter()
    gl.ylocator = mticker.FixedLocator(np.arange(-30,45,15))
    gl.yformatter = LatitudeFormatter()

fig.colorbar(im, cax=cbar_axis, orientation='vertical')

plt.show()

In [ ]:
variable_to_plot = 'specific_humidity'
plev_to_plot = 850

times_to_plot = np.arange(
    np.datetime64(init_date) + np.timedelta64(1, 'D'),
    np.datetime64(init_date) + np.timedelta64(30, 'D'),
    np.timedelta64(4, 'D')
)

fig = plt.figure(figsize=(12, len(times_to_plot)))
gs = GridSpec(len(times_to_plot), 2, figure=fig, width_ratios=[30, 1])
gs.update(top=1, bottom=0, left=0, right=0.8, hspace=0.1, wspace=0.05)

proj = ccrs.PlateCarree(central_longitude=180)
data_crs = ccrs.PlateCarree()

ax = [
    fig.add_subplot(gs[i, 0], projection=proj) 
    for i in range(len(times_to_plot))
]

cbar_axis = fig.add_subplot(gs[:, 1])

cf_data = xarray_utils.add_cyclic_point(
    input_data[variable_to_plot].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
).sel(lat=slice(-30,30)).stats.standardize(dim=['lat', 'lon'])

quiv_u_data = xarray_utils.add_cyclic_point(
    input_data['u_component_of_wind'].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
).sel(lat=slice(-30,30))

quiv_v_data = xarray_utils.add_cyclic_point(
    input_data['v_component_of_wind'].coord_funcs.sel(level=plev_to_plot),
    dim='lon'
).sel(lat=slice(-30,30))

ax[0].set_title(f"Event starting {np.datetime64(init_date).astype('M8[ms]').astype('O').strftime('%d%^b%Y')}", fontsize=16)
for index, time in enumerate(times_to_plot):
    im = ax[index].contourf(
        cf_data.lon,
        cf_data.lat,
        cf_data.sel(datetime=time),
        # levels=np.arange(-125, 125, 25),
        levels=21,
        transform=data_crs,
        cmap='coolwarm',
        norm=mcolors.CenteredNorm(vcenter=0),
    )
    ax[index].add_feature(cfeature.COASTLINE, lw=1)
    ax[index].text(
        s=f"{(time-times_to_plot[0]).astype('timedelta64[D]')}", 
        x=1-0.015, 
        y=0.9,
        transform=ax[index].transAxes,
        fontsize=12,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none')
        )

    arrow_spacing = 8
    ax[index].quiver(
        quiv_u_data.lon[::2*arrow_spacing],
        quiv_v_data.lat[::arrow_spacing],
        quiv_u_data.sel(datetime=time)[::arrow_spacing, ::2*arrow_spacing].values,
        quiv_v_data.sel(datetime=time)[::arrow_spacing, ::2*arrow_spacing].values,
        transform=data_crs,
        width=0.002,
        # scale=200
    )

    minlon = central_longitude - 180
    maxlon = central_longitude + 180
    ax[index].set_extent([minlon, maxlon, -30, 30], data_crs)
    gl = ax[index].gridlines(
        crs=data_crs,
        draw_labels=(True if index == len(times_to_plot) - 1 else False),
        linewidth=1,
        color="gray",
        alpha=0.75,
        linestyle="-",
        zorder=15

    )
    gl.right_labels = False
    gl.top_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180,180,60))
    gl.xformatter = LongitudeFormatter()
    gl.ylocator = mticker.FixedLocator(np.arange(-30,45,15))
    gl.yformatter = LatitudeFormatter()

fig.colorbar(im, cax=cbar_axis, orientation='vertical')

plt.show()